# Adaptive Feedback Loop Based Cognitive SOC: Threat Classification
This Jupyter Notebook implements the machine learning threat classification module for the Cognitive SOC.

## Methodology Overview:
1. **Data Sources:**
    - ** TIP:** Real-world Indicators of Compromise (IOC) from `Data/IOC.json` (N = 4,200)
2. **Feature Engineering:** Extracts **15 features** (3 indicator type one-hot, 5 threat category one-hot, and 7 scalar features).
3. **Model:** Random Forest Classifier with `class_weight='balanced'`.
4. **Evaluation:** Classification reports, Confusion Matrix, MTTR trends, and validation checks.

In [5]:
import os
import json
import sqlite3
import pandas as pd
import numpy as np
import pickle
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set base paths
BASE_DIR = os.getcwd()
IOC_JSON_PATH = os.path.join(BASE_DIR, "Data", "IOC.json")
MODEL_OUT_PATH = os.path.join(BASE_DIR, "Data", "ioc_rf_model.pkl")
ENCODER_OUT_PATH = os.path.join(BASE_DIR, "Data", "label_encoder.pkl")

print(f"Workspace directory: {BASE_DIR}")

Workspace directory: /content


In [6]:
# 1. Load & Preprocess Cyfirma real dataset (`IOC.json`)

with open(IOC_JSON_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} raw indicators from IOC.json.")

ext_def = "extension-definition--8e1c4b7d-9a2f-4d63-b5e0-3c7a1f9d2b44"
cyfirma_records = []

for item in raw_data:
    name = item.get("name", "")
    pattern = item.get("pattern", "")
    confidence = item.get("confidence", 70)
    created_str = item.get("created", "2026-06-05T00:08:20.181Z")
    modified_str = item.get("modified", "2026-06-05T00:08:20.181Z")

    # Determine indicator_type
    ioc_type = "Hash"
    if "ipv4-addr" in pattern or "ipv6-addr" in pattern:
        ioc_type = "IP"
    elif "domain-name" in pattern:
        ioc_type = "Domain"

    # Extract properties
    props = item.get("extensions", {}).get(ext_def, {}).get("properties", {})
    score = props.get("exposure_score", 5)
    verdict = props.get("verdict", "Suspicious")
    country = props.get("country_code", "US")
    asn_owner = props.get("asn_owner", "")
    vendors = props.get("security_vendors", {})

    # Threat Category mapping
    threat_cat = "Other"
    desc_lower = item.get("description", "").lower()
    vendor_values_str = " ".join(str(v).lower() for v in vendors.values())
    if "c2" in desc_lower or "c2" in vendor_values_str or "command and control" in desc_lower:
        threat_cat = "C2"
    elif "phishing" in desc_lower or "phishing" in vendor_values_str:
        threat_cat = "Phishing"
    elif "exploit" in desc_lower or "vulnerability" in desc_lower:
        threat_cat = "Exploit"
    elif "malware" in desc_lower or "trojan" in desc_lower or "botnet" in desc_lower or "malicious" in vendor_values_str:
        threat_cat = "Malware"

    # Scalar features
    tip_reputation_score = (score * 10) / 100.0
    confidence_score = confidence / 100.0

    # Source diversity count
    source_diversity_count = np.log1p(len(vendors))

    # Active days count
    try:
        dt_created = datetime.strptime(created_str.split(".")[0] + "Z", "%Y-%m-%dT%H:%M:%SZ")
        dt_modified = datetime.strptime(modified_str.split(".")[0] + "Z", "%Y-%m-%dT%H:%M:%SZ")
        days = max(1, (dt_modified - dt_created).days)
    except:
        days = 1
    active_days_count = np.log1p(days)

    # Geolocation risk
    geo_risk = 1.0
    if country in ["RU", "CN", "KP", "IR", "BY"]:
        geo_risk = 2.0
    elif country in ["US", "CA", "GB", "DE", "FR", "JP"]:
        geo_risk = 0.0
    geolocation_risk = geo_risk / 2.0

    # ASN reputation
    asn_rep = 0.5
    if "google" in asn_owner.lower() or "amazon" in asn_owner.lower() or "cloudflare" in asn_owner.lower():
        asn_rep = 0.1
    elif not asn_owner:
        asn_rep = 0.8
    asn_reputation = asn_rep

    first_seen_age_days = active_days_count * 0.8

    # Label mapping L
    if score == 10:
        severity = "Critical"
    elif score in [6, 7]:
        severity = "High"
    elif score <= 3 or verdict in ["Benign", "Unknown"]:
        severity = "Low"
    else:
        severity = "Medium"

    cyfirma_records.append({
        "id": item.get("id"),
        "ioc_value": name,
        "indicator_type": ioc_type,
        "threat_category": threat_cat,
        "tip_reputation_score": tip_reputation_score,
        "confidence_score": confidence_score,
        "source_diversity_count": source_diversity_count,
        "active_days_count": active_days_count,
        "geolocation_risk": geolocation_risk,
        "asn_reputation": asn_reputation,
        "first_seen_age_days": first_seen_age_days,
        "severity": severity,
        "source": "Cyfirma"
    })

df_cyfirma = pd.DataFrame(cyfirma_records)
print(f"Extracted {len(df_cyfirma)} Cyfirma records.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/Data/IOC.json'

In [ ]:
## 3. Merge & Sample Target Distribution (N = 3,000, using only Cyfirma data)

combined_df = df_cyfirma.copy()

target_dist = {
    'Low': 1350,
    'Medium': 900,
    'High': 540,
    'Critical': 210
}

sampled_dfs = []
for label, count in target_dist.items():
    pool = combined_df[combined_df['severity'] == label]
    sampled = pool.sample(count, replace=(len(pool) < count), random_state=42)
    sampled_dfs.append(sampled)

study_df = pd.concat(sampled_dfs).sample(frac=1.0, random_state=42).reset_index(drop=True)

# Normalize scalar features
for col in ['source_diversity_count', 'active_days_count', 'first_seen_age_days']:
    study_df[col] = (study_df[col] - study_df[col].min()) / (study_df[col].max() - study_df[col].min())

print("Final sampled distribution:")
print(study_df['severity'].value_counts())

In [ ]:
## 4. One-Hot Encoding and Split

X_cat = pd.get_dummies(study_df[['indicator_type', 'threat_category']], dtype=float)
X_num = study_df[['tip_reputation_score', 'confidence_score', 'source_diversity_count', 'active_days_count', 'geolocation_risk', 'asn_reputation', 'first_seen_age_days']]
X = pd.concat([X_cat, X_num], axis=1)

expected_features = [
    'indicator_type_Domain', 'indicator_type_Hash', 'indicator_type_IP',
    'threat_category_C2', 'threat_category_Exploit', 'threat_category_Malware', 'threat_category_Phishing', 'threat_category_Other',
    'tip_reputation_score', 'confidence_score', 'source_diversity_count', 'active_days_count', 'geolocation_risk', 'asn_reputation', 'first_seen_age_days'
]

X = X.reindex(columns=expected_features, fill_value=0.0)
y = study_df['severity']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.20, stratify=y_encoded, random_state=42)
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape  : {X_val.shape}")

In [ ]:
## 5. Random Forest Training & Evaluation

clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_val)
print(f"Validation Accuracy: {accuracy_score(y_val, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=le.classes_))

In [ ]:
## 6. Telemetry Plots

# Plot 1: 4x4 Confusion Matrix (Using Real ML Output)
# This dynamically calculates the matrix based on your actual model predictions
cm = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            annot_kws={'size': 12, 'weight': 'bold'})
plt.ylabel('Actual Class', fontweight='bold')
plt.xlabel('Predicted Class', fontweight='bold')
plt.title('Fig 3. Random Forest Confusion Matrix', fontweight='bold')
plt.show()

# Save model & label encoder
with open(MODEL_OUT_PATH, 'wb') as f:
    pickle.dump(clf, f)
with open(ENCODER_OUT_PATH, 'wb') as f:
    pickle.dump(le, f)

print("Model saved to Data/ioc_rf_model.pkl")
print("Label encoder saved to Data/label_encoder.pkl")